# Settings

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/Islamic_Stories_Project"

LORA_PATH = f"{BASE_DIR}/models/allam-arastories-lora"
CSV_PATH = f"{BASE_DIR}/ALLaM_output.csv"


for label, p in [
    ("LORA", LORA_PATH),
    ("Output CSV", CSV_PATH),
]:
    print(("✓" if os.path.exists(p) else "✗ MISSING"), label, "->", p)

Mounted at /content/drive
✓ LORA -> /content/drive/MyDrive/Islamic_Stories_Project/models/allam-arastories-lora
✗ MISSING Output CSV -> /content/drive/MyDrive/Islamic_Stories_Project/ALLaM_output.csv


In [2]:
!pip install -q -U transformers accelerate peft bitsandbytes sentence-transformers pandas==2.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 146.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 46.0 MB/s eta 0:00:00


# Load ALLaM

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

MODEL_NAME = "humain-ai/ALLaM-7B-Instruct-preview"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit QLoRA config — identical to your original notebook
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Base model
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
)

# Attach your trained LoRA adapter (no retraining)
ft_model = PeftModel.from_pretrained(base, LORA_PATH).eval()
print("✓ ALLaM + LoRA loaded")

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✓ ALLaM + LoRA loaded


# Functions

In [6]:
# ============================================================
# Functions — ALLaM generation + summary + query + CSV saving
# ============================================================

import os
import csv
from datetime import datetime

# نفس system prompt القديم
SYSTEM_PROMPT = "أنت علام، كاتب قصص عربية للأطفال يركز على القيم الإسلامية والعبرة في النهاية."


# ============================================================
# 1. Story generation
# ============================================================

def generate_story_allam(
    age,
    moral,
    topic,
    place=None,
    end_of_story=None,
    dialogue=None,
    num_characters=None,
    country=None,
    season=None,
    activity=None,
    emotion=None,
    plot_twist=None,
    max_new_tokens=400
):
    """
    Generate Arabic children's story using the loaded ALLaM + LoRA model.
    Uses the same original prompt and generation settings.
    """

    features = [
        f"- عُمر الطفل/الطفلة: {age} سنة",
        f"- القيمة الإسلامية (العبرة): {moral}",
        f"- الموضوع العام للقصة: {topic}",
    ]

    if place:
        features.append(f"- مكان أحداث القصة: {place}")
    if country:
        features.append(f"- الدولة: {country}")
    if season:
        features.append(f"- الفصل: {season}")
    if activity:
        features.append(f"- النشاط الرئيسي في القصة: {activity}")
    if num_characters:
        features.append(f"- عدد الشخصيات الأساسية: {num_characters}")
    if emotion:
        features.append(f"- الشعور العام في القصة: {emotion}")

    if dialogue is not None:
        features.append(
            "- تضمين حوار بين الشخصيات"
            if dialogue
            else "- تقليل الحوار والتركيز على السرد"
        )

    if plot_twist is not None:
        features.append(
            "- تحتوي على حبكة مفاجِئة في النهاية"
            if plot_twist
            else "- بدون حبكة مفاجِئة"
        )

    if end_of_story:
        features.append(f"- شكل نهاية القصة المطلوب: {end_of_story}")

    user_prompt = (
        "أريد منك أن تكتب قصة عربية للأطفال بناءً على المواصفات التالية:\n\n"
        + "\n".join(features)
        + "\n\nشروط مهمة:\n"
        "- استخدم لغة عربية مبسطة وممتعة تناسب الأطفال.\n"
        "- اجعل القصة مترابطة وواضحة.\n"
        "- في النهاية، اكتب سطرًا يبدأ بكلمة: \"العبرة:\" ثم قدّم العبرة بشكل صريح وواضح."
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_prompt
        },
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(device)

    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
        )

    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    story = tokenizer.decode(gen_ids, skip_special_tokens=True)

    return story.strip()


# ============================================================
# 2. Extract moral
# ============================================================

def extract_moral(text):
    """
    Extract only the explicit moral line after 'العبرة:'.
    """
    marker = "العبرة:"

    if marker not in text:
        return ""

    after_marker = text.split(marker, 1)[1].strip()
    lines = [line.strip() for line in after_marker.splitlines() if line.strip()]

    if not lines:
        return ""

    moral_line = lines[0]
    moral_line = moral_line.strip().strip('"').strip("“”").strip("«»")

    return moral_line.strip()


# ============================================================
# 3. Summary for retrieval
# ============================================================

def summarize_story_for_retrieval(story_text, topic, moral):
    """
    Generate short value-focused summary using ALLaM.
    Same summary prompt and settings.
    """

    prompt = f"""
لديك قصة عربية للأطفال، وأريد تلخيصها لغرض البحث عن حديث نبوي مناسب.

الموضوع: {topic}
القيمة الإسلامية التي أدخلها المستخدم: {moral}

القصة:
{story_text}

اكتب ملخصًا قصيرًا جدًا من جملة واحدة فقط.
ركّز على:
- الحدث الأساسي في القصة
- القيمة الإسلامية أو الأخلاقية
- العبرة المناسبة

تجنب ذكر التفاصيل الجانبية مثل الأسماء، المكان، الفصل، أو الوصف الطويل.
لا تكتب عنوانًا ولا شرحًا. اكتب الملخص فقط.
""".strip()

    messages = [
        {
            "role": "system",
            "content": "أنت مساعد يلخص قصص الأطفال العربية لغرض مطابقة القصة مع حديث نبوي مناسب."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(device)

    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    summary = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    summary = summary.replace("الملخص:", "").replace("ملخص:", "").strip()
    summary = summary.strip().strip('"').strip("“”").strip("«»")

    lines = [line.strip() for line in summary.splitlines() if line.strip()]
    if lines:
        summary = lines[0]

    return summary.strip()


# ============================================================
# 4. Query builders
# ============================================================

def build_query_with_summary(topic, moral, story_summary=None):
    """
    Query with summary.
    The summary is stored inside the query only.
    """
    q = (
        f"القيمة الإسلامية: {topic}\n"
        f"العبرة من القصة: {moral}"
    )

    if story_summary:
        q += f"\nملخص القصة: {story_summary}"

    return q


def build_query_no_summary(topic, moral):
    """
    Query without summary.
    """
    return (
        f"القيمة الإسلامية: {topic}\n"
        f"العبرة من القصة: {moral}"
    )


# ============================================================
# 5. CSV saving
# ============================================================

GEN_COLUMNS = [
    "sample_id",
    "timestamp",

    "model_name",

    "input_age",
    "input_topic",
    "input_moral",
    "input_place",
    "input_country",
    "input_season",
    "input_activity",
    "input_emotion",
    "input_dialogue",
    "input_plot_twist",
    "input_end_of_story",

    "generated_story",
    "extracted_moral",

    "query_with_summary",
    "query_no_summary",
]


def append_generation_csv(path, row):
    """
    Append one generation result to CSV.
    """
    new_file = not os.path.exists(path)

    parent = os.path.dirname(os.path.abspath(path))
    if parent:
        os.makedirs(parent, exist_ok=True)

    with open(path, "w" if new_file else "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=GEN_COLUMNS)

        if new_file:
            writer.writeheader()

        writer.writerow(row)

    return row


def get_next_sample_id(path):
    """
    Auto sample_id based on existing CSV rows.
    """
    if os.path.exists(path):
        try:
            old_df = pd.read_csv(path)
            return len(old_df) + 1
        except Exception:
            return 1

    return 1


print("✓ ALLaM generation functions ready")

✓ ALLaM generation functions ready


## أمثلة

مثال الصدق

In [8]:
# ============================================================
# Run ALLaM generation + summary + queries + save
# ============================================================

sample_id = get_next_sample_id(CSV_PATH)

inp = {
    "age": 8,
    "topic": "الصدق",
    "moral": "أهمية الصدق",
    "place": "المدرسة",
    "country": "السعودية",
    "season": "الخريف",
    "activity": "وقت اللعب",
    "emotion": "حماسي",
    "dialogue": True,
    "plot_twist": True,
    "end_of_story": "اعتراف صادق",
    "num_characters": None,
}

print("=" * 80)
print("Running ALLaM generation")
print("Sample ID:", sample_id)
print("=" * 80)

story = generate_story_allam(
    age=inp["age"],
    moral=inp["moral"],
    topic=inp["topic"],
    place=inp.get("place"),
    country=inp.get("country"),
    season=inp.get("season"),
    activity=inp.get("activity"),
    emotion=inp.get("emotion"),
    dialogue=inp.get("dialogue"),
    plot_twist=inp.get("plot_twist"),
    end_of_story=inp.get("end_of_story"),
    num_characters=inp.get("num_characters"),
    max_new_tokens=400,
)

extracted_moral = extract_moral(story)

story_summary = summarize_story_for_retrieval(
    story_text=story,
    topic=inp["topic"],
    moral=inp["moral"]
)

query_with_summary = build_query_with_summary(
    topic=inp["topic"],
    moral=inp["moral"],
    story_summary=story_summary
)

query_no_summary = build_query_no_summary(
    topic=inp["topic"],
    moral=inp["moral"]
)

out_row = {
    "sample_id": sample_id,
    "timestamp": datetime.now().isoformat(timespec="seconds"),

    "model_name": "ALLaM",

    "input_age": inp.get("age"),
    "input_topic": inp.get("topic"),
    "input_moral": inp.get("moral"),
    "input_place": inp.get("place"),
    "input_country": inp.get("country"),
    "input_season": inp.get("season"),
    "input_activity": inp.get("activity"),
    "input_emotion": inp.get("emotion"),
    "input_dialogue": inp.get("dialogue"),
    "input_plot_twist": inp.get("plot_twist"),
    "input_end_of_story": inp.get("end_of_story"),

    "generated_story": story,
    "extracted_moral": extracted_moral,

    "query_with_summary": query_with_summary,
    "query_no_summary": query_no_summary,
}

append_generation_csv(CSV_PATH, out_row)

print("\n" + "=" * 80)
print("Generated Story")
print("=" * 80)
print(story)

print("\n" + "=" * 80)
print("Extracted Moral")
print("=" * 80)
print(extracted_moral)

print("\n" + "=" * 80)
print("Query WITH summary")
print("=" * 80)
print(query_with_summary)

print("\n" + "=" * 80)
print("Query WITHOUT summary")
print("=" * 80)
print(query_no_summary)

print("\nSaved to:")
print(CSV_PATH)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Running ALLaM generation
Sample ID: 1


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Generated Story
في بلدة صغيرة جميلة بالسعودية، كان هناك طفل اسمه علي. علي كان في الثامنة من عمره، وكان يذهب إلى المدرسة كل يوم حيث كان يتعلم الصدق والأمانة.

ذات يوم في الخريف، كان علي يلعب مع أصدقائه في فترة الاستراحة بعد المدرسة. كانوا يلعبون لعبة "الصيد" حيث يحاولون الإمساك ببعضهم البعض. علي كان يحاول بجد أن يمسك صديقه محمد، وكان محمد يهرب منه باستمرار.

فجأة، رأى علي شخصًا غريبًا يحاول التسلل إلى المدرسة. دون تفكير، قال علي الحقيقة لأصدقائه: "محمد، لا تهرب. هذا الشخص يحاول الدخول إلى المدرسة بشكل غير قانوني."

محمد، بعد أن أدرك جدية الوضع، أخبر علي أنه كان خائفًا من أن يكون الشخص الغريب معلمًا أو شخصًا من المدرسة. علي كان قلقًا جدًا على صديقه، لكنه كان يعلم أن الصدق هو الشيء الصحيح.

بعد التحقق من الأمر، تبين أن محمد كان مخطئًا. الشخص الغريب كان مجرد بائع متجول جاء لبيع بعض الحلويات. علي ومحمد كلاهما شعر بالخجل من ما حدث، لكن علي اعترف لمحمد أنه كان يجب أن يصدق كلامه في البداية.

العبرة: الصدق هو ما يجعلنا أقوى ويحمي من حولنا. 

علي ومحمد تعلما درسًا قيمًا في ذلك اليوم، وأهمية الص

مثال الصيام

In [11]:
# ============================================================
# Run ALLaM generation + summary + queries + save
# ============================================================

sample_id = get_next_sample_id(CSV_PATH)

inp = {
    "age": 8,
    "topic": "الصيام",
    "moral": "فضل الصيام",
    "place": "المنزل",
    "country": "السعودية",
    "season": "الشتاء",
    "activity": "الاستعداد لشهر رمضان",
    "emotion": "روحاني",
    "dialogue": True,
    "plot_twist": False,
    "end_of_story": "طمأنينة",
    "num_characters": None,
}

print("=" * 80)
print("Running ALLaM generation")
print("Sample ID:", sample_id)
print("=" * 80)

story = generate_story_allam(
    age=inp["age"],
    moral=inp["moral"],
    topic=inp["topic"],
    place=inp.get("place"),
    country=inp.get("country"),
    season=inp.get("season"),
    activity=inp.get("activity"),
    emotion=inp.get("emotion"),
    dialogue=inp.get("dialogue"),
    plot_twist=inp.get("plot_twist"),
    end_of_story=inp.get("end_of_story"),
    num_characters=inp.get("num_characters"),
    max_new_tokens=400,
)

extracted_moral = extract_moral(story)

story_summary = summarize_story_for_retrieval(
    story_text=story,
    topic=inp["topic"],
    moral=inp["moral"]
)

query_with_summary = build_query_with_summary(
    topic=inp["topic"],
    moral=inp["moral"],
    story_summary=story_summary
)

query_no_summary = build_query_no_summary(
    topic=inp["topic"],
    moral=inp["moral"]
)

out_row = {
    "sample_id": sample_id,
    "timestamp": datetime.now().isoformat(timespec="seconds"),

    "model_name": "ALLaM",

    "input_age": inp.get("age"),
    "input_topic": inp.get("topic"),
    "input_moral": inp.get("moral"),
    "input_place": inp.get("place"),
    "input_country": inp.get("country"),
    "input_season": inp.get("season"),
    "input_activity": inp.get("activity"),
    "input_emotion": inp.get("emotion"),
    "input_dialogue": inp.get("dialogue"),
    "input_plot_twist": inp.get("plot_twist"),
    "input_end_of_story": inp.get("end_of_story"),

    "generated_story": story,
    "extracted_moral": extracted_moral,

    "query_with_summary": query_with_summary,
    "query_no_summary": query_no_summary,
}

append_generation_csv(CSV_PATH, out_row)

print("\n" + "=" * 80)
print("Generated Story")
print("=" * 80)
print(story)

print("\n" + "=" * 80)
print("Extracted Moral")
print("=" * 80)
print(extracted_moral)

print("\n" + "=" * 80)
print("Query WITH summary")
print("=" * 80)
print(query_with_summary)

print("\n" + "=" * 80)
print("Query WITHOUT summary")
print("=" * 80)
print(query_no_summary)

print("\nSaved to:")
print(CSV_PATH)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Running ALLaM generation
Sample ID: 2


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Generated Story
في قرية صغيرة في المملكة العربية السعودية، حيث الشتاء يكسو الأرض بردائه الأبيض، كان هناك طفل اسمه يوسف، في الثامنة من عمره، يستعد بشغف لشهر رمضان الكريم. يوسف كان يحب الصيام منذ صغره، وكان ينتظر قدوم رمضان بفارغ الصبر ليختبر شعوره بالفرحة والروحانية.

يوسف كان يجلس كل يوم مع والدته في المطبخ، يتعلم منها كيف يُعد الأطباق الرمضانية الشهية. كان يقول لها: "أمي، كيف أشعر بالسعادة في رمضان؟" أجابت الأم بابتسامة: "الصيام، يا يوسف، ليس فقط عن الامتناع عن الطعام والشراب، بل هو فرصة لتجربة شعور العطاء والإحسان، وفرصة لتقوية علاقتنا بالله."

مع اقتراب رمضان، كان يوسف يقضي معظم وقته في قراءة القرآن والدعاء، متمنيًا أن يكون صيامه مقبولاً. في أحد الأيام، وبينما كان يوسف يستيقظ مبكرًا ليصلي الفجر، شعر بفرح غامر يغمره. قال يوسف لنفسه: "أشعر بالرضا والسكينة، كأنني أخيرًا فهمت معنى الصيام."

في ليلة من الليالي، قبل أن يبدأ الصيام في اليوم التالي، كان يوسف يجلس مع عائلته، يتناولون وجبة الإفطار. قال يوسف بابتسامة: "لقد تعلمت الكثير يا أمي، لقد فهمت أن الصيام ليس فقط عن الجوع والعطش، بل هو

مثال الصلاة

In [12]:
# ============================================================
# Run ALLaM generation + summary + queries + save
# ============================================================

sample_id = get_next_sample_id(CSV_PATH)

inp = {
    "age": 8,
    "topic": "الصلاة",
    "moral": "أهمية الصلاة",
    "place": "المسجد",
    "country": "السعودية",
    "season": "الصيف",
    "activity": "الذهاب للمسجد",
    "emotion": "حنون",
    "dialogue": True,
    "plot_twist": False,
    "end_of_story": "نهاية سعيدة",
    "num_characters": None,
}

print("=" * 80)
print("Running ALLaM generation")
print("Sample ID:", sample_id)
print("=" * 80)

story = generate_story_allam(
    age=inp["age"],
    moral=inp["moral"],
    topic=inp["topic"],
    place=inp.get("place"),
    country=inp.get("country"),
    season=inp.get("season"),
    activity=inp.get("activity"),
    emotion=inp.get("emotion"),
    dialogue=inp.get("dialogue"),
    plot_twist=inp.get("plot_twist"),
    end_of_story=inp.get("end_of_story"),
    num_characters=inp.get("num_characters"),
    max_new_tokens=400,
)

extracted_moral = extract_moral(story)

story_summary = summarize_story_for_retrieval(
    story_text=story,
    topic=inp["topic"],
    moral=inp["moral"]
)

query_with_summary = build_query_with_summary(
    topic=inp["topic"],
    moral=inp["moral"],
    story_summary=story_summary
)

query_no_summary = build_query_no_summary(
    topic=inp["topic"],
    moral=inp["moral"]
)

out_row = {
    "sample_id": sample_id,
    "timestamp": datetime.now().isoformat(timespec="seconds"),

    "model_name": "ALLaM",

    "input_age": inp.get("age"),
    "input_topic": inp.get("topic"),
    "input_moral": inp.get("moral"),
    "input_place": inp.get("place"),
    "input_country": inp.get("country"),
    "input_season": inp.get("season"),
    "input_activity": inp.get("activity"),
    "input_emotion": inp.get("emotion"),
    "input_dialogue": inp.get("dialogue"),
    "input_plot_twist": inp.get("plot_twist"),
    "input_end_of_story": inp.get("end_of_story"),

    "generated_story": story,
    "extracted_moral": extracted_moral,

    "query_with_summary": query_with_summary,
    "query_no_summary": query_no_summary,
}

append_generation_csv(CSV_PATH, out_row)

print("\n" + "=" * 80)
print("Generated Story")
print("=" * 80)
print(story)

print("\n" + "=" * 80)
print("Extracted Moral")
print("=" * 80)
print(extracted_moral)

print("\n" + "=" * 80)
print("Query WITH summary")
print("=" * 80)
print(query_with_summary)

print("\n" + "=" * 80)
print("Query WITHOUT summary")
print("=" * 80)
print(query_no_summary)

print("\nSaved to:")
print(CSV_PATH)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Running ALLaM generation
Sample ID: 3


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Generated Story
في قرية صغيرة جميلة بالمملكة العربية السعودية، حيث الأجواء الصيفية الحارة، كان هناك طفل يُدعى يوسف. كان يوسف في الثامنة من عمره، وكان دائمًا يُفكر في كيفية قضاء يومه بأفضل طريقة.

في صباح أحد الأيام، بينما كان يوسف يلعب مع أصدقائه خارج المنزل، سمع صوت الأذان من المسجد القريب. قال يوسف لأصدقائه: "دعونا نذهب إلى المسجد ونصلي معًا."

كان يوسف متحمسًا للغاية للذهاب إلى المسجد. وصلوا إلى هناك، وكان المسجد مليئًا بالأطفال الذين جاءوا للصلاة. يوسف شعر بسعادة غامرة، وكان ينظر حوله متعجبًا من جمال المسجد وروعة ألوانه.

بعد الصلاة، جلس يوسف وأصدقاؤه في المسجد يتحدثون عن أهمية الصلاة في الإسلام. يوسف قال بحماس: "الصلاة تُقربنا من الله وتجعلنا نشعر بالسلام الداخلي."

وفي تلك اللحظة، شعر يوسف بحضن دافئ من والده، الذي جاء ليأخذه من المسجد. قال يوسف لوالده: "أبي، لقد شعرت بالسلام والراحة في المسجد اليوم، الصلاة تُشعرنا بالهدوء والسعادة."

ابتسم والد يوسف وقال: "هذا صحيح يا يوسف. الصلاة تُعتبر عماد الدين، وهي تجعلنا أقوى وأكثر ارتباطًا بالله."

العبرة: "الصلاة ليست مجرد عبادة نؤديها، 

In [13]:
import os
import pandas as pd

print("CSV_PATH:", CSV_PATH)
print("Exists:", os.path.exists(CSV_PATH))

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print("Rows:", len(df))
    display(df.tail())
else:
    print("File does not exist yet.")

CSV_PATH: /content/drive/MyDrive/Islamic_Stories_Project/ALLaM_output.csv
Exists: True
Rows: 3


,sample_id,timestamp,model_name,input_age,input_topic,input_moral,input_place,input_country,input_season,input_activity,input_emotion,input_dialogue,input_plot_twist,input_end_of_story,generated_story,extracted_moral,query_with_summary,query_no_summary
0,1,2026-05-22T04:24:17,ALLaM,8,الصدق,أهمية الصدق,المدرسة,السعودية,الخريف,وقت اللعب,حماسي,True,True,اعتراف صادق,في بلدة صغيرة جميلة بالسعودية، كان هناك طفل اس...,الصدق هو ما يجعلنا أقوى ويحمي من حولنا.,القيمة الإسلامية: الصدق\nالعبرة من القصة: أهمي...,القيمة الإسلامية: الصدق\nالعبرة من القصة: أهمي...
1,2,2026-05-22T04:33:55,ALLaM,8,الصيام,فضل الصيام,المنزل,السعودية,الشتاء,الاستعداد لشهر رمضان,روحاني,True,False,طمأنينة,في قرية صغيرة في المملكة العربية السعودية، حيث...,رمضان يعلمنا أن الصيام ليس فقط عن الامتناع عن ...,القيمة الإسلامية: الصيام\nالعبرة من القصة: فضل...,القيمة الإسلامية: الصيام\nالعبرة من القصة: فضل...
2,3,2026-05-22T04:37:07,ALLaM,8,الصلاة,أهمية الصلاة,المسجد,السعودية,الصيف,الذهاب للمسجد,حنون,True,False,نهاية سعيدة,في قرية صغيرة جميلة بالمملكة العربية السعودية،...,الصلاة ليست مجرد عبادة نؤديها، بل هي مصدر السل...,القيمة الإسلامية: الصلاة\nالعبرة من القصة: أهم...,القيمة الإسلامية: الصلاة\nالعبرة من القصة: أهم...
